# 非量化Flash Attention介绍

上一节我们了解了本章的整体规划。本节正式介绍**非量化**Flash Attention算子：它是什么、与量化版本的差异、算子规格、计算流程，以及在Ascend C上的实现思路。本节只介绍概念与思路，**不涉及具体代码实现**——具体实现将在后续章节展开。

本节学习大纲如下：

- 量化与非量化Flash Attention
- 非量化Flash Attention算子规格
- 计算流程概述
- Ascend C实现思路
- 小结

---

## 1. 量化与非量化Flash Attention

Flash Attention是Transformer中注意力计算的高性能实现，通过分块计算（Tiling）与在线Softmax（Online Softmax）减少HBM访存次数，加速长序列训练与推理。根据输入数据类型与计算精度，Flash Attention可分为两类：

- **非量化Flash Attention**：Q、K、V 均为高精度浮点类型（FP16 或 BF16），Softmax 在 FP32 上累加以保证数值精度，全过程不涉及量化/反量化操作。**本章聚焦此类。**
- **量化Flash Attention**：Q、K（甚至V）被量化为低比特整数或低精度浮点（如 INT8、MXFP8 等），计算时配合量化参数反量化，以进一步降低访存带宽、提升吞吐，但实现更复杂、对精度控制要求更高。

二者的核心差异如下：

| 维度 | 非量化FA | 量化FA |
|------|---------|--------|
| 输入dtype | FP16 / BF16 | INT8 / MXFP8 等低精度 |
| Softmax精度 | FP32累加 | 需配合反量化 |
| 实现复杂度 | 较低，概念清晰 | 较高，需管理量化参数 |
| 带宽/性能 | 标准 | 更省带宽、更高吞吐 |
| 精度 | 高 | 取决于量化策略 |

> 本章只讨论**非量化**场景，即 Q、K、V 均为 FP16（half）输入。BF16 的实现思路与之类似，仅数据类型不同。

---

## 2. 非量化Flash Attention算子规格

### 2.1 数学表达式

非量化Flash Attention的计算公式为：

```
O = softmax(Q × K^T × scale) × V
```

其中 scale = 1 / √D，D 为头维度，用于防止点积过大导致softmax梯度消失。

### 2.2 输入输出

- 输入：q、k、v，均为 half（FP16）类型。
- 输出：attn_out，half 类型。
- 属性：softmax_scale（缩放因子，可选，默认 1/√D）。

### 2.3 形状与布局

- q: [B, QS, N, D]，B为批次大小，QS为Query序列长度，N为注意力头数，D为头维度。
- k: [B, KVS, N, D]，KVS为Key/Value序列长度。
- v: [B, KVS, N, Dv]，Dv为Value头维度（通常 Dv = D）。
- attn_out: [B, QS, N, Dv]。
- 数据布局（Format）：BSND。

以 B=1, QS=128, KVS=256, N=1, D=64, Dv=64 作为演示示例。

<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 650px; border: 1px solid #ddd;">
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子类型 (OpType)</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" > FlashAttnCustom</td>
  </tr>
  <tr>
    <td rowspan="4" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">shape</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">q</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(1, 128, 1, 64)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">BSND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">k</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(1, 256, 1, 64)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">BSND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">v</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(1, 256, 1, 64)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">half</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">BSND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">attn_out</td>
    <td style="padding: 8px;">(1, 128, 1, 64)</td>
    <td style="padding: 8px;">half</td>
    <td style="padding: 8px;">BSND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子属性</td>
    <td style="padding: 8px;">softmax_scale</td>
    <td style="padding: 8px;">-</td>
    <td style="padding: 8px;">float</td>
    <td style="padding: 8px;">-</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>  

---

## 3. 计算流程概述

非量化Flash Attention采用**分块计算 + 在线Softmax**，单核计算流程如下：

```
┌─────────────────────────────────────────────────────────────┐
│  KVS方向循环（核内）                                         │
│  ┌──────────┐    ┌──────────────┐    ┌──────────┐          │
│  │  QK^T    │───▶│  在线Softmax  │───▶│   PV     │──▶ 累加O│
│  │ (Cube)   │    │  (Vector)    │    │ (Cube)   │          │
│  └──────────┘    └──────────────┘    └──────────┘          │
│   Q×K^T=S        更新m,l           P×V=ΔO                   │
└─────────────────────────────────────────────────────────────┘
```

每个KVS块的迭代包含三个阶段（算法层面，不涉及具体代码）：

1. **QK^T**：计算 Q × K^T × scale，得到注意力分数S。
2. **在线Softmax**：在片上存储中更新运行状态（行最大值 m、行指数和 l），得到概率矩阵P。通过维护这两个运行状态，无需看到完整的S矩阵即可逐步逼近标准Softmax的结果。
3. **PV**：计算 P × V，将结果累加到输出O。

所有KVS块处理完毕后，对O按行除以 l 完成最终归一化。

> Flash Attention的输出与标准Attention**完全一致**，是精确算法而非近似。本节只描述算法流程，具体的片上存储管理、API调用与Tiling切分将在后续实现章节展开。

---

## 4. Ascend C实现思路

在Ascend C中实现非量化Flash Attention，核心是**Cube计算（Matmul）与Vector计算的协同**。本节仅介绍实现思路，不展开具体代码。

### 4.1 Cube与Vector协同

- **QK^T 与 PV** 是两次矩阵乘，属于Cube计算，使用Matmul高阶API完成。
- **在线Softmax** 涉及Max、Exp、Sum等逐元素与归约运算，属于Vector计算，在UB（Unified Buffer）上执行。

### 4.2 两个Matmul对象

单个核函数中需要创建两个Matmul对象：

- **Matmul 1（QK^T）**：A = Q，B = K^T（B矩阵设为转置），C = S（float）。
- **Matmul 2（PV）**：A = P，B = V，C = ΔO（float）。

### 4.3 Tiling思路（概念）

- **多核切分**：沿QS（Query序列长度）方向将Q切分给多个核并行处理。
- **核内循环**：K、V沿KVS（Key/Value序列长度）方向分块，每个核在KVS方向循环多次，通过在线Softmax累加结果。
- **Buffer规划**：QK^T结果S、Softmax结果P、PV累加结果O等中间数据需要合理规划存储位置（Global Memory的workspace与片上UB）。

### 4.4 数据类型配合

- 输入输出：half（FP16）。
- 中间计算：QK^T结果与Softmax运算使用float（FP32），以保证数值精度。
- P矩阵在送入第二次Matmul（PV）之前，需由float转换回half。

> 具体的Tiling结构体定义、Host侧Tiling实现、Kernel侧核函数代码将在后续章节详细展开。

---

## 5. 小结

本节介绍了非量化Flash Attention的概念、算子规格、计算流程与Ascend C实现思路。要点回顾：

- 非量化FA的Q/K/V为FP16/BF16，Softmax在FP32累加，全过程不涉及量化/反量化。
- 算子规格：输入q[B,QS,N,D]、k[B,KVS,N,D]、v[B,KVS,N,Dv]（half, BSND），输出attn_out[B,QS,N,Dv]（half, BSND），属性softmax_scale。
- 计算流程为 QK^T → 在线Softmax → PV → 累加 → 归一化，输出与标准Attention精确一致。
- 实现上需Cube（Matmul）与Vector协同，使用两个Matmul对象，并沿QS多核切分、KVS核内循环。

下一节将进入具体的代码实现。

---

## 课后练习

请根据本节课程学习内容完成以下题目进行自测。

1. 非量化Flash Attention 与 量化Flash Attention 的主要区别是什么？

    A. 非量化版使用了近似算法，精度更低
    
    B. 非量化版Q/K/V保持FP16/BF16，不涉及量化/反量化，Softmax在FP32累加
    
    C. 量化版的输出与标准Attention不同
    
    D. 非量化版只能用于推理，不能用于训练

2. 非量化Flash Attention的Softmax通常在什么精度下累加以保证数值精度？

    A. FP16

    B. FP32

    C. INT8

    D. INT4

3. 在Ascend C的非量化FA实现中，QK^T与PV两次矩阵乘属于哪类计算？

    A. 都是Vector计算

    B. 都是Cube计算，使用Matmul高阶API

    C. QK^T是Cube，PV是Vector

    D. 两者都用ReduceSum实现

4. 非量化Flash Attention的输出与标准Attention相比：

    A. 是近似结果，存在数值误差

    B. 完全相同，是精确算法

    C. 仅在FP16精度下相同

    D. 在长序列场景下不同

5. 非量化FA实现中，P矩阵在送入第二次Matmul（PV）之前，需要做什么？

    A. 从FP32转换为FP16

    B. 从FP16转换为FP32

    C. 做量化

    D. 不需要任何类型转换

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/02.02_answer.txt
